# Stretch Robot – Evaluation
Run `stretch3_common.py` first, then choose a section below.

In [ ]:
%load_ext autoreload
%autoreload 2
%run stretch3_common.py

## Load Model

In [ ]:
# model = PPO.load("ppo_discrete_lift_arm_k3_ms200_lr3e4_g098_entCoef005_fullTable_F1_steps_81920")

## Simple Step-by-Step Evaluation
Visual step loop with per-action probability printout.

In [ ]:
# import torch
# from stable_baselines3.common.utils import obs_as_tensor

# model = PPO.load("ppo_discrete_lift_arm_k3_ms200_lr3e4_g098_entCoef005_fullTable_F2_steps_112640")

# action_mode = "discrete"
# allowed_parts = "lift_arm"
# control_hold = 3
# current_mode = "easy"
# headless = False
# fixed_y = -0.5962
# success_thresh = 0.06
# lift_start_pos = 0.62

# eval_env = DummyVecEnv([make_env(
#     action_mode=action_mode,
#     allowed_parts=allowed_parts,
#     control_hold=control_hold,
#     default_mode = current_mode,
#     headless=headless,
#     fixed_y=fixed_y,
#     success_thresh=success_thresh,
#     lift_start_pos=lift_start_pos,
#     run_name="eval"
#     )])

# # RESET FIRST
# obs = eval_env.reset()

# ACTION_NAMES = ["lift_down", "lift_up", "noop", "arm_in", "arm_out"]

# for i in range(200):
#     # --- Get action probabilities ---
#     obs_tensor = obs_as_tensor(obs, model.policy.device)
#     with torch.no_grad():
#         distribution = model.policy.get_distribution(obs_tensor)
#         probs = distribution.distribution.probs[0].cpu().numpy()

#     action, _ = model.predict(obs, deterministic=True)
#     obs, rewards, dones, infos = eval_env.step(action)

#     info = infos[0]
#     prob_str = "  ".join([f"{ACTION_NAMES[j]}={probs[j]:.2%}" for j in range(5)])
#     print(
#         f"i={i:02d}, act={info['action_id']}({info['action_name']}), "
#         f"rew={rewards[0]:+.4f}, d={info['distance']:.4f}, lift={info['lift_pos']:.3f}, ee_z={info['ee_z']:.3f} | {prob_str}"
#     )

#     if dones[0]:
#         print("episode return:", info["episode"]["r"])
#         print("episode length:", info["episode"]["l"])
#         break

## Automated Sweep
Run N_POSITIONS × N_REPS_PER_POS trials and export summary + details CSVs.

In [ ]:
# ── Automated Evaluation Sweep ──────────────────────────────────────────────
# N_POSITIONS lift heights × N_REPS_PER_POS trials each → aggregated results.
# Outputs to eval_results/{model_name}_{timestamp}_summary.csv
#                          {model_name}_{timestamp}_steps.csv

import os
import numpy as np
import pandas as pd
import torch
from datetime import datetime
from stable_baselines3.common.utils import obs_as_tensor
from stable_baselines3.common.vec_env import DummyVecEnv

# ── Config ───────────────────────────────────────────────────────────────────
MODEL_PATH      = "ppo_discrete_lift_arm_k3_ms200_lr3e4_g098_entCoef005_fullTable_F2_steps_112640"
N_REPS_PER_POS  = 3            # trials per position (to average out physics variance)
MIN_LIFT        = 0.6
MAX_LIFT        = 1.1
lift_positions = np.arange(MIN_LIFT, MAX_LIFT + 0.001, 0.01)  # +0.001 to include 1.1
N_POSITIONS = len(lift_positions)
MAX_STEPS       = 200
ACTION_MODE ="discrete"
ALLOWED_PARTS ="lift_arm"
CONTROL_HOLD =3
DEFAULT_MODE ="hard"
SUCCESS_THRESH  = 0.06
FIXED_Y         = -0.5
HEADLESS        = True
LIFT_START_RANDOM = False
OUT_DIR         = "eval_results"
ACTION_NAMES    = ["lift_down", "lift_up", "noop", "arm_in", "arm_out"]
MOE = 0.01

# ── Setup ─────────────────────────────────────────────────────────────────────
os.makedirs(OUT_DIR, exist_ok=True)
timestamp  = datetime.now().strftime("%Y%m%d_%H%M%S")
model_name = os.path.basename(MODEL_PATH)
model = PPO.load(MODEL_PATH)

total_trials   = N_POSITIONS * N_REPS_PER_POS
all_summaries, all_steps = [], []
trial_count = 0

vec_env = DummyVecEnv([make_env(
    action_mode=ACTION_MODE,
    allowed_parts=ALLOWED_PARTS,
    control_hold=CONTROL_HOLD,
    default_mode=DEFAULT_MODE,
    headless=HEADLESS,
    fixed_y=FIXED_Y,
    success_thresh=SUCCESS_THRESH,
    lift_start_pos=MIN_LIFT,
    lift_start_random=LIFT_START_RANDOM,
    run_name="eval",
)])

# ── Sweep ─────────────────────────────────────────────────────────────────────
for pos_idx, lift_pos in enumerate(lift_positions):
    lift_pos = round(float(lift_pos), 4)
    # print(f"target_start_lift_pos: {lift_pos}")
    
    rep_results = []

    for rep in range(N_REPS_PER_POS):
        trial_count += 1
        vec_env.envs[0].unwrapped.lift_start_pos = lift_pos
        obs = vec_env.reset()
        actual_start_lift_pos = obs["observation"][0][0]
        # print(f"actual_start_lift_pos: {actual_start_lift_pos}")

        step_rows = []
        for i in range(MAX_STEPS):
            obs_tensor = obs_as_tensor(obs, model.policy.device)
            with torch.no_grad():
                probs = model.policy.get_distribution(obs_tensor).distribution.probs[0].cpu().numpy()

            # After vec_env.step() fires dones[0]=True, DummyVecEnv auto-resets — so obs["desired_goal"][0][1] is the next episode's y, not this one's. Fix: capture before the step:
            obj_y = round(float(obs["desired_goal"][0][1]), 4) 

            action, _ = model.predict(obs, deterministic=True)
            obs, rewards, dones, infos = vec_env.step(action)
            info = infos[0]

            step_rows.append({
                "lift_start_pos": lift_pos, "rep": rep, "step": i,
                "mode":           DEFAULT_MODE,
                "success_thresh": SUCCESS_THRESH,
                "action_id": info["action_id"], "action_name": info["action_name"],
                "reward":         round(float(rewards[0]), 4),
                "distance":       round(float(info["distance"]), 4),
                "lift_pos":       round(float(info["lift_pos"]), 4),
                "obj_y": obj_y, 
                "obj_z":          round(float(info["goal_z"]), 4),
                "ee_z":           round(float(info["ee_z"]), 4),
                **{f"prob_{ACTION_NAMES[j]}": round(float(probs[j]), 4) for j in range(len(ACTION_NAMES))},
            })

            if dones[0]:
                ep = info.get("episode", {})
                
                final_d            = info["distance"]
                final_ee_z  = info["ee_z"]
                obj_z       = info["goal_z"]

                is_success_strict  = bool(info.get("is_success", 0))
                is_success_lenient = final_d < SUCCESS_THRESH   # ← real task metric (no ee_z gate)
                is_success_moe = final_d < SUCCESS_THRESH and abs(final_ee_z - obj_z) <= MOE

                min_d              = min(r["distance"] for r in step_rows)
                max_ee_z           = max(r["ee_z"] for r in step_rows)
                termination        = "success" if is_success_strict else "early_term"

                summary = {
                    "lift_start_pos":     lift_pos,
                    "rep":                rep,
                    "mode":               DEFAULT_MODE,
                    "success_thresh":     SUCCESS_THRESH,
                    "termination":        termination,
                    "is_success":         is_success_strict,         # training metric (d<0.15 AND ee_z>=apple_z)
                    "is_success_lenient": is_success_lenient,        # eval metric (d<0.15 only)
                    "is_success_moe": is_success_moe,
                    "final_distance":     round(final_d, 4),
                    "min_distance":       round(min_d, 4),           # ← closest the arm got
                    "max_ee_z":           round(max_ee_z, 4),        # ← how high arm got
                    "episode_return":     ep.get("r", rewards[0]),
                    "episode_length":     ep.get("l", i + 1),
                    "final_ee_z": round(info["ee_z"], 4),
                    "obj_y":        round(step_rows[0]["obj_y"], 4),
                    "obj_z":      round(info["goal_z"], 4),
                }
                break
        else:
            last = step_rows[-1] if step_rows else {}
            min_d    = min(r["distance"] for r in step_rows) if step_rows else float("nan")
            max_ee_z = max(r["ee_z"] for r in step_rows) if step_rows else float("nan")
            summary = {
                "lift_start_pos":     lift_pos,
                "rep":                rep,
                "mode":               DEFAULT_MODE,
                "success_thresh":     SUCCESS_THRESH,
                "obj_y":              step_rows[0]["obj_y"] if step_rows else float("nan"),  # ← add
                "obj_z":              last.get("obj_z", float("nan")),                        # ← add
                "termination":        "timeout",
                "is_success":         False,
                "is_success_lenient": False,
                "is_success_moe":     False,                                                  # ← add
                "final_distance":     round(last.get("distance", float("nan")), 4),
                "final_ee_z":         round(last.get("ee_z", float("nan")), 4),              # ← add
                "min_distance":       round(min_d, 4),
                "max_ee_z":           round(max_ee_z, 4),
                "episode_return":     sum(r["reward"] for r in step_rows),
                "episode_length":     MAX_STEPS,
            }

        rep_results.append(summary)
        all_summaries.append(summary)
        all_steps.extend(step_rows)

    # Per-position aggregate print
    sr   = np.mean([r["is_success"] for r in rep_results]) * 100
    avgd = np.mean([r["final_distance"] for r in rep_results])
    print(f"[{pos_idx+1:02d}/{N_POSITIONS}] lift={lift_pos:.4f} | "
          f"success={sr:.0f}% ({sum(r['is_success'] for r in rep_results)}/{N_REPS_PER_POS}) | "
          f"avg_d={avgd:.4f}")

# ── Save ──────────────────────────────────────────────────────────────────────
summary_path = f"{OUT_DIR}/{model_name}_{timestamp}_summary.csv"
steps_path   = f"{OUT_DIR}/{model_name}_{timestamp}_details.csv"
pd.DataFrame(all_summaries).to_csv(summary_path, index=False, float_format="%.4f")
pd.DataFrame(all_steps).to_csv(steps_path, index=False, float_format="%.4f")

# Aggregate by lift position
df = pd.DataFrame(all_summaries)
agg = df.groupby("lift_start_pos").agg(
    success_rate        =("is_success", "mean"),
    success_rate_lenient=("is_success_lenient", "mean"),   # ← real task metric
    success_rate_moe     = ("is_success_moe", "mean"),    # ← add
    avg_min_distance    =("min_distance", "mean"),          # ← best reach per trial
    avg_final_distance  =("final_distance", "mean"),
    avg_max_ee_z        =("max_ee_z", "mean"),
    avg_return          =("episode_return", "mean"),
    avg_length          =("episode_length", "mean"),
    n_success           = ("is_success", "sum"),      # cleaner than dict
    n_early_term        = ("termination", lambda x: (x == "early_term").sum()),
    n_timeout           = ("termination", lambda x: (x == "timeout").sum()),
).reset_index()

print(f"\n{'='*60}")
print(f"Sweep done. {N_POSITIONS} positions × {N_REPS_PER_POS} reps = {total_trials} total trials.")
print(f"Overall success rate: {df['is_success'].mean()*100:.1f}%")
print(f"Avg final_d:  {df['final_distance'].mean():.4f}")
print(f"\nPer-position summary:")
print(agg.to_string(index=False))
print(f"\nSummary -> {summary_path}")
print(f"Steps   -> {steps_path}")

vec_env.close()


### Post-Sweep MOE Analysis (optional)

In [ ]:
# import pandas as pd

# MARGIN_OF_ERROR = 0.01
# moe_col = f"is_success_moe_{MARGIN_OF_ERROR}"

# detail_file_name = f"ppo_discrete_lift_arm_k3_ms200_lr3e4_g098_entCoef005_fullTable_F2_steps_112640_20260419_182603_details"
# summary_file_name = f"ppo_discrete_lift_arm_k3_ms200_lr3e4_g098_entCoef005_fullTable_F2_steps_112640_20260419_182603_summary"

# steps_df = pd.read_csv(f"eval_results/{detail_file_name}.csv")
# summary_df = pd.read_csv(f"eval_results/{summary_file_name}.csv")

# # Get the terminal step for each episode (lift_start_pos + rep)
# final_steps = (
#     steps_df
#     .groupby(["lift_start_pos", "rep"])
#     .last()
#     .reset_index()
# )

# # Compute new is_success_lenient: d < 0.15 AND ee_z within ±0.1 of obj_z
# final_steps[moe_col] = (
#     (final_steps["distance"] < 0.15) &
#     (abs(final_steps["ee_z"] - final_steps["obj_z"]) <= MARGIN_OF_ERROR)
# )

# # print(final_steps)
# # print(final_steps[["lift_start_pos", "rep", "obj_z", "ee_z", moe_col]])

# # Merge back into your summary CSV if needed
# summary_df = summary_df.merge(
#     final_steps[["lift_start_pos", "rep", "ee_z", moe_col]],
#     on=["lift_start_pos", "rep"],
#     how="left"
# )

# # print(summary_df)

# # Aggregate by position
# agg = summary_df.groupby("lift_start_pos").agg(**{
#     "success_rate":("is_success", "mean"),
#     "success_lenient":("is_success_lenient", "mean"),
#     f"success_moe_{MARGIN_OF_ERROR}":(moe_col, "mean"),
#     "avg_ee_z":("ee_z", "mean"),
#     "avg_obj_z":("obj_z", "mean"),
# }).reset_index()

# # print(agg.to_string(index=False))
# # # Total across all positions
# for label, col in [
#     ("success_rate",    "is_success"),
#     ("success_lenient", "is_success_lenient"),
#     (f"success_moe_{MARGIN_OF_ERROR}", moe_col),
# ]:
#     n = summary_df[col].sum()
#     total = len(summary_df)
#     print(f"\nTotal {label}: {n/total*100:.1f}%  ({n:.0f} / {total})")


### Re-test Failed Positions from a Previous Run (optional)

In [ ]:
# # ── Load failed positions from a previous run ─────────────────────────────
# PREV_SUMMARY = "eval_results/ppo_discrete_lift_arm_k3_ms200_lr3e4_g098_entCoef005_fullTable_F1_steps_81920_20260417_143617_summary.csv"

# prev_df = pd.read_csv(PREV_SUMMARY)
# failed_positions = (
#     prev_df[prev_df["is_success_lenient"] == False]["lift_start_pos"]
#     .unique()
# )

# lift_positions = np.array(sorted(failed_positions))
# N_POSITIONS    = len(lift_positions)
# print(f"Re-testing {N_POSITIONS} positions that had at least one failure")
# print(lift_positions)

### Commented eval-loop variant (legacy)

In [ ]:
# # try:
# ep_r = 0.0

# for i in range(100):
#     action, _ = model.predict(obs, deterministic=True)
#     obs, rewards, dones, infos = eval_env.step(action)
#     # print(action)
    
#     ep_r += float(rewards[0])
    
#     info = infos[0]   # because DummyVecEnv wraps it
#     print(
#         f"i={i}, act={info['action_id']}({info['action_name']}), "
#         f"rew={rewards[0]:+.4f}, d={info['distance']:.4f}, "
#         f"lift={info['lift_pos']:.3f}, "   # ADD THIS
#         f"arm={info['arm_pos']:.3f}, d_arm={info['delta_arm']:+.4f}"
#     )
    
#     if dones[0]:
#         if "episode" in info:
#             print("episode return (Monitor):", info["episode"]["r"])
#             print("episode length (Monitor):", info["episode"]["l"])
            
#         # optional: also see why it ended
#         print("terminated/truncated:", info.get("TimeLimit.truncated", None))
#         break
        
# # finally:
# #     # IMPORTANT: shuts down the Mujoco server process started inside make_env().
# #     eval_env.close()

## Graphs – Monitor Progress

In [ ]:
# plot_monitor_progress("monitor_train.csv")

## End / Teardown

In [ ]:

# # IMPORTANT: shuts down the Mujoco server process started inside make_env().
# if eval_env.envs[0].is_running():
#     eval_env.envs[0].close()
# eval_env.close()

In [ ]:
# # 1) Stop the current headless sim (if running)
# if sim.is_running():
#     sim.stop()